# Desafio Técnico — Nível 1
Prevenção à Lavagem de Dinheiro — tratamento de dados, regras determinísticas e análise com LLM.

Fluxo do notebook:
1. Carga dos dados
2. Limpeza e explicação dos problemas encontrados
3. Normalização para BRL
4. Agregações (volume por cliente, contagem por canal)
5. Regra 1 — Fracionamento
6. Regra 2 — Valor atípico
7. Validação das regras
8. Análise com LLM (parecer estruturado) + comparação de dois prompts

In [1]:
import json
import pandas as pd

with open("../dados/dados_nivel_1.json", encoding="utf-8") as f:
    raw = json.load(f)

taxa_cambio = raw["taxa_cambio_usd_brl"]
df = pd.DataFrame(raw["operacoes"])
print(f"Taxa de câmbio USD->BRL: {taxa_cambio}")
print(f"Shape bruto: {df.shape}")
df.head(10)

Taxa de câmbio USD->BRL: 5.4
Shape bruto: (20, 9)


        id cliente_id        data  valor moeda   canal                   tipo          contraparte observacao
0  OP-0001    CLI-A-1  2026-03-09  18100   BRL     pix  transferencia_enviada   Alfa Comercio LTDA           
1  OP-0002    CLI-A-1  2026-03-09  17300   BRL     pix  transferencia_enviada   Alfa Comercio LTDA           
2  OP-0003    CLI-A-1  2026-03-09  18800   BRL     ted  transferencia_enviada     Beta Servicos ME           
3  OP-0004    CLI-A-1  2026-03-21   3300   BRL  boleto              pagamento   Gama Distribuidora           
4  OP-0005    CLI-A-2  2026-03-14  25900   BRL     ted  transferencia_enviada    Delta Transportes           
5  OP-0006    CLI-A-2  2026-03-14  27000   BRL     ted  transferencia_enviada    Delta Transportes           
6  OP-0007    CLI-A-3  2026-03-05  17200   BRL     pix  transferencia_enviada  Epsilon Consultoria           
7  OP-0008    CLI-A-3  2026-03-05  15200   BRL     pix  transferencia_enviada  Epsilon Consultoria           
8  OP-0009

## Parte A — Tratamento e regras

### O que encontrei de errado nos dados

Antes de aplicar qualquer regra, inspecionei nulos e duplicatas:

- **Linha duplicada**: `OP-0007` aparece duas vezes, com todos os campos idênticos (mesmo cliente, mesma data, mesmo valor, mesma contraparte). É claramente um registro replicado pelo sistema de origem, não duas operações reais — se eu não removesse, o cliente `CLI-A-3` seria indevidamente sinalizado pela Regra 1 (a soma passaria de R$ 48.500 para R$ 65.700 com 4 lançamentos "diferentes"). **Decisão:** removi a duplicata mantendo a primeira ocorrência (`drop_duplicates(subset="id")`).
- **Data ausente**: `OP-0017` tem `data: null` e uma observação explícita ("data não capturada pelo sistema"). Não posso inventar uma data — isso distorceria a Regra 1, que depende de agrupar por dia. **Decisão:** mantive a operação no DataFrame (ela ainda é uma transação real e entra nos totais e na Regra 2), mas ela fica automaticamente fora de qualquer agrupamento por data (`data_dt` vira `NaT`) e por isso nunca participa da Regra 1. Isso é intencional e mais seguro do que preencher com uma data arbitrária, que poderia tanto criar quanto esconder um alerta.
- **Moeda mista**: `OP-0013` está em USD, não em BRL. Todas as regras de valor (fracionamento, atípico) precisam estar na mesma unidade — **decisão:** normalizei tudo para BRL usando a taxa fixa fornecida no próprio arquivo (`valor_brl = valor * taxa` quando `moeda == "USD"`).
- Não encontrei valores negativos, tipos de operação inconsistentes ou outliers de digitação (ex.: valor com centenas de casas) nesta base — mas deixei a checagem de nulos genérica (`df.isna().sum()`) como primeira linha de defesa caso apareçam em bases maiores (Nível 2).

In [2]:
print("Nulos por coluna:")
print(df.isna().sum())
print()
dup_mask = df.duplicated(keep=False)
print(f"Linhas envolvidas em duplicata exata: {dup_mask.sum()}")
df[dup_mask]

Nulos por coluna:
id             0
cliente_id     0
data           1
valor          0
moeda          0
canal          0
tipo           0
contraparte    0
observacao     0
dtype: int64

Linhas envolvidas em duplicata exata: 2


        id cliente_id        data  valor moeda canal                   tipo          contraparte observacao
6  OP-0007    CLI-A-3  2026-03-05  17200   BRL   pix  transferencia_enviada  Epsilon Consultoria           
9  OP-0007    CLI-A-3  2026-03-05  17200   BRL   pix  transferencia_enviada  Epsilon Consultoria           

In [3]:
# Remove duplicata exata de id, mantendo a primeira ocorrência
df = df.drop_duplicates(subset="id", keep="first").reset_index(drop=True)
print(f"Shape após remover duplicata: {df.shape}")

Shape após remover duplicata: (19, 9)


### Normalização para BRL

In [4]:
def valor_em_brl(row):
    if row["moeda"] == "USD":
        return row["valor"] * taxa_cambio
    return row["valor"]

df["valor_brl"] = df.apply(valor_em_brl, axis=1)
df["data_dt"] = pd.to_datetime(df["data"], errors="coerce")  # OP-0017 vira NaT, de propósito

df[["id", "cliente_id", "valor", "moeda", "valor_brl", "data", "data_dt"]]

         id cliente_id  valor moeda  valor_brl        data    data_dt
0   OP-0001    CLI-A-1  18100   BRL    18100.0  2026-03-09 2026-03-09
1   OP-0002    CLI-A-1  17300   BRL    17300.0  2026-03-09 2026-03-09
2   OP-0003    CLI-A-1  18800   BRL    18800.0  2026-03-09 2026-03-09
3   OP-0004    CLI-A-1   3300   BRL     3300.0  2026-03-21 2026-03-21
4   OP-0005    CLI-A-2  25900   BRL    25900.0  2026-03-14 2026-03-14
5   OP-0006    CLI-A-2  27000   BRL    27000.0  2026-03-14 2026-03-14
6   OP-0007    CLI-A-3  17200   BRL    17200.0  2026-03-05 2026-03-05
7   OP-0008    CLI-A-3  15200   BRL    15200.0  2026-03-05 2026-03-05
8   OP-0009    CLI-A-3  16100   BRL    16100.0  2026-03-05 2026-03-05
9   OP-0010    CLI-A-4   3800   BRL     3800.0  2026-03-03 2026-03-03
10  OP-0011    CLI-A-4   5100   BRL     5100.0  2026-03-11 2026-03-11
11  OP-0012    CLI-A-4   5800   BRL     5800.0  2026-03-18 2026-03-18
12  OP-0013    CLI-A-4  12000   USD    64800.0  2026-03-24 2026-03-24
13  OP-0014    CLI-A

### Agregações
- Volume total transacionado por cliente (em BRL)
- Quantidade de operações por canal

In [5]:
volume_por_cliente = (
    df.groupby("cliente_id")["valor_brl"].sum().sort_values(ascending=False)
)
volume_por_cliente

cliente_id
CLI-A-4    79500.0
CLI-A-1    57500.0
CLI-A-2    52900.0
CLI-A-3    48500.0
CLI-A-5    16900.0
CLI-A-6    10200.0

In [6]:
qtd_por_canal = df["canal"].value_counts()
qtd_por_canal

canal
pix        8
ted        5
boleto     3
cartao     2
especie    1

### Regra 1 — Fracionamento

Sinaliza **cliente + data** quando, naquele dia, houve 3 ou mais operações cuja soma
ultrapassa R$ 50.000,00 e nenhuma operação isolada atinge R$ 20.000,00.

Isso é puro cálculo (contagem, soma, comparação com limite) — não peço isso à LLM,
só rodo em pandas.

In [7]:
def aplica_regra_fracionamento(df):
    """Retorna o set de chaves (cliente_id, data) sinalizadas pela Regra 1.
    Só considera linhas com data válida (NaT nunca entra em um grupo de fracionamento)."""
    flagged = set()
    df_com_data = df.dropna(subset=["data_dt"])
    for (cliente, data), grupo in df_com_data.groupby(["cliente_id", "data_dt"]):
        if len(grupo) >= 3 and grupo["valor_brl"].sum() > 50_000 and (grupo["valor_brl"] < 20_000).all():
            flagged.add((cliente, data))
    return flagged

chaves_fracionamento = aplica_regra_fracionamento(df)
df["flag_fracionamento"] = df.apply(
    lambda r: pd.notna(r["data_dt"]) and (r["cliente_id"], r["data_dt"]) in chaves_fracionamento,
    axis=1,
)
print("Clientes/datas sinalizados pela Regra 1:")
for cli, data in chaves_fracionamento:
    print(f"  - {cli} em {data.date()}")
df[df["flag_fracionamento"]][["id", "cliente_id", "data", "valor_brl"]]

Clientes/datas sinalizados pela Regra 1:
  - CLI-A-1 em 2026-03-09


        id cliente_id        data  valor_brl
0  OP-0001    CLI-A-1  2026-03-09    18100.0
1  OP-0002    CLI-A-1  2026-03-09    17300.0
2  OP-0003    CLI-A-1  2026-03-09    18800.0

### Regra 2 — Valor atípico

Sinaliza a **operação** cujo valor em BRL seja superior a 5× a mediana dos valores
daquele mesmo cliente. Só se aplica a clientes com 4 ou mais operações.

In [8]:
def aplica_regra_valor_atipico(df):
    flags = pd.Series(False, index=df.index)
    for cliente, grupo in df.groupby("cliente_id"):
        if len(grupo) >= 4:
            mediana = grupo["valor_brl"].median()
            limite = 5 * mediana
            flags.loc[grupo.index] = grupo["valor_brl"] > limite
    return flags

df["flag_valor_atipico"] = aplica_regra_valor_atipico(df)
df[df["flag_valor_atipico"]][["id", "cliente_id", "valor_brl"]]

         id cliente_id  valor_brl
12  OP-0013    CLI-A-4    64800.0

### Validação das regras

Preciso mostrar que a Regra 1 captura o caso que deveria capturar (**CLI-A-1**) e
**não** captura um caso parecido que não se enquadra.

`CLI-A-2` é o contraste perfeito: no mesmo dia (14/03) tem duas operações que somam
R$ 52.900 (passa de R$ 50 mil) e nenhuma isolada chega a R$ 20 mil — mas são só
**2** operações, não 3. A regra exige contagem mínima de 3, então `CLI-A-2`
corretamente **não** é sinalizado, mesmo "parecendo" um caso de fracionamento à
primeira vista.

In [9]:
caso_positivo = df[df["cliente_id"] == "CLI-A-1"][["id", "data", "valor_brl", "flag_fracionamento"]]
caso_negativo = df[df["cliente_id"] == "CLI-A-2"][["id", "data", "valor_brl", "flag_fracionamento"]]

print("Caso que DEVE ser capturado (CLI-A-1, 3 operações, soma > 50k, nenhuma >= 20k):")
display(caso_positivo)

print("\nCaso parecido que NÃO deve ser capturado (CLI-A-2, só 2 operações, soma também > 50k):")
display(caso_negativo)

assert caso_positivo["flag_fracionamento"].any(), "Regra 1 falhou em capturar CLI-A-1"
assert not caso_negativo["flag_fracionamento"].any(), "Regra 1 sinalizou um falso positivo em CLI-A-2"
print("\nValidação OK: Regra 1 se comporta como esperado nos dois casos.")

Caso que DEVE ser capturado (CLI-A-1, 3 operações, soma > 50k, nenhuma >= 20k):
        id        data  valor_brl  flag_fracionamento
0  OP-0001  2026-03-09    18100.0                True
1  OP-0002  2026-03-09    17300.0                True
2  OP-0003  2026-03-09    18800.0                True
3  OP-0004  2026-03-21     3300.0               False

Caso parecido que NÃO deve ser capturado (CLI-A-2, só 2 operações, soma também > 50k):
        id        data  valor_brl  flag_fracionamento
4  OP-0005  2026-03-14    25900.0               False
5  OP-0006  2026-03-14    27000.0               False

Validação OK: Regra 1 se comporta como esperado nos dois casos.


## Parte B — Análise com LLM

Escolhi o cliente **CLI-A-4**, sinalizado pela Regra 2 (a operação `OP-0013`, uma
remessa internacional de US$ 12.000 recebida de `Zeta Importacao`, é 5x+ maior que
a mediana das operações desse cliente).

Importante: a LLM **não** recalcula limite nem soma nada — todo o cálculo já foi
feito acima em pandas. A LLM só recebe o resumo pronto e escreve um parecer
interpretativo, com saída estruturada e validada.

⚠️ **Sobre a execução desta célula**: o notebook chama uma LLM real (Gemini/Groq/
outro provedor gratuito) via variável de ambiente `LLM_API_KEY`. Para manter o
notebook executável sem expor nenhuma chave e sem depender de rede no ambiente de
avaliação, a função `chamar_llm()` abaixo tem um modo `mock=True` que simula a
chamada (mesma interface, mesma latência de contrato) — troque para `mock=False`
com sua chave configurada em `.env` antes da entrega final para gerar o parecer
com uma LLM de verdade. Isso está registrado em `docs/DECISOES.md`.

In [10]:
import os, time, json as _json

def montar_resumo_cliente(df, cliente_id):
    sub = df[df["cliente_id"] == cliente_id]
    return {
        "cliente_id": cliente_id,
        "qtd_operacoes": int(len(sub)),
        "volume_total_brl": float(sub["valor_brl"].sum()),
        "canais_usados": sub["canal"].value_counts().to_dict(),
        "contrapartes": sorted(sub["contraparte"].unique().tolist()),
        "operacoes_flagueadas": sub[sub["flag_valor_atipico"] | sub["flag_fracionamento"]][
            ["id", "data", "valor_brl", "canal", "tipo", "contraparte", "observacao"]
        ].to_dict("records"),
    }

resumo_cli_a4 = montar_resumo_cliente(df, "CLI-A-4")
resumo_cli_a4

{'cliente_id': 'CLI-A-4', 'qtd_operacoes': 4, 'volume_total_brl': 79500.0, 'canais_usados': {'cartao': 1, 'boleto': 1, 'pix': 1, 'ted': 1}, 'contrapartes': ['Alfa Comercio LTDA', 'Beta Servicos ME', 'Gama Distribuidora', 'Zeta Importacao'], 'operacoes_flagueadas': [{'id': 'OP-0013', 'data': '2026-03-24', 'valor_brl': 64800.00000000001, 'canal': 'ted', 'tipo': 'transferencia_recebida', 'contraparte': 'Zeta Importacao', 'observacao': 'remessa internacional'}]}

In [11]:
PROMPT_V1 = f"""Você é analista de PLD (Prevenção à Lavagem de Dinheiro) de um banco.
Analise o cliente abaixo e dê sua opinião sobre o risco.

Dados: {resumo_cli_a4}
"""

PROMPT_V2 = f"""Você é analista sênior de PLD (Prevenção à Lavagem de Dinheiro).
Você recebe abaixo o resumo AGREGADO (já calculado) de um cliente e as operações
que o motor de regras determinísticas já sinalizou. Você NÃO deve recalcular nada
— apenas interpretar.

Responda ESTRITAMENTE em JSON, sem texto fora do JSON, com exatamente estas chaves:
- "nivel_risco": um de "baixo", "medio", "alto"
- "tipologia_suspeita": string curta (ex.: "estruturação", "comércio internacional atípico", "sem indício claro")
- "red_flags": lista de strings, cada uma um indício objetivo observado nos dados
- "justificativa": 2-4 frases explicando o raciocínio, citando os campos que pesaram

Resumo do cliente:
{resumo_cli_a4}
"""

print("=== PROMPT V1 (aberto, sem formato exigido) ===")
print(PROMPT_V1)
print("\n=== PROMPT V2 (formato estruturado, papel definido, proibido recalcular) ===")
print(PROMPT_V2)

=== PROMPT V1 (aberto, sem formato exigido) ===
Você é analista de PLD (Prevenção à Lavagem de Dinheiro) de um banco.
Analise o cliente abaixo e dê sua opinião sobre o risco.

Dados: {'cliente_id': 'CLI-A-4', 'qtd_operacoes': 4, 'volume_total_brl': 79500.0, 'canais_usados': {'cartao': 1, 'boleto': 1, 'pix': 1, 'ted': 1}, 'contrapartes': ['Alfa Comercio LTDA', 'Beta Servicos ME', 'Gama Distribuidora', 'Zeta Importacao'], 'operacoes_flagueadas': [{'id': 'OP-0013', 'data': '2026-03-24', 'valor_brl': 64800.00000000001, 'canal': 'ted', 'tipo': 'transferencia_recebida', 'contraparte': 'Zeta Importacao', 'observacao': 'remessa internacional'}]}


=== PROMPT V2 (formato estruturado, papel definido, proibido recalcular) ===
Você é analista sênior de PLD (Prevenção à Lavagem de Dinheiro).
Você recebe abaixo o resumo AGREGADO (já calculado) de um cliente e as operações
que o motor de regras determinísticas já sinalizou. Você NÃO deve recalcular nada
— apenas interpretar.

Responda ESTRITAMENTE em

In [12]:
def chamar_llm(prompt, mock=True):
    """Chama a LLM configurada em LLM_API_KEY. Com mock=True, simula a resposta
    para manter o notebook executável sem chave/rede no ambiente de avaliação —
    trocar para mock=False antes da entrega final (ver docs/DECISOES.md)."""
    inicio = time.time()
    if mock or not os.getenv("LLM_API_KEY"):
        # --- SIMULAÇÃO: mesma interface de uma chamada real, para o notebook rodar fim-a-fim ---
        texto = _json.dumps({
            "nivel_risco": "medio",
            "tipologia_suspeita": "comércio internacional atípico",
            "red_flags": [
                "Recebimento único em USD muito acima do padrão do próprio cliente (5x+ a mediana)",
                "Contraparte associada a operações de outros clientes no mesmo período (Zeta Importacao)",
                "Ausência de histórico prévio de operações internacionais para este cliente",
            ],
            "justificativa": "O cliente concentra a maior parte do seu volume em pagamentos e transferências domésticas de baixo valor; a única exceção é um recebimento internacional expressivo, sinalizado pela Regra 2. Isso por si só não configura lavagem, mas justifica revisão humana antes de liberar novas remessas.",
        }, ensure_ascii=False)
        tokens_entrada = len(prompt) // 4  # aproximação simples de tokens
        tokens_saida = len(texto) // 4
        time.sleep(0.05)
    else:
        # --- CHAMADA REAL (exemplo com API compatível OpenAI, ex.: Groq/OpenRouter) ---
        import requests
        resp = requests.post(
            "https://api.groq.com/openai/v1/chat/completions",
            headers={"Authorization": f"Bearer {os.getenv('LLM_API_KEY')}"},
            json={
                "model": "llama-3.3-70b-versatile",
                "messages": [{"role": "user", "content": prompt}],
                "temperature": 0,
            },
            timeout=30,
        )
        data = resp.json()
        texto = data["choices"][0]["message"]["content"]
        tokens_entrada = data["usage"]["prompt_tokens"]
        tokens_saida = data["usage"]["completion_tokens"]

    duracao = time.time() - inicio
    return {
        "texto": texto,
        "tokens_entrada": tokens_entrada,
        "tokens_saida": tokens_saida,
        "tempo_segundos": round(duracao, 3),
    }

resultado_v2 = chamar_llm(PROMPT_V2, mock=True)
resultado_v2

{'texto': '{"nivel_risco": "medio", "tipologia_suspeita": "comércio internacional atípico", "red_flags": ["Recebimento único em USD muito acima do padrão do próprio cliente (5x+ a mediana)", "Contraparte associada a operações de outros clientes no mesmo período (Zeta Importacao)", "Ausência de histórico prévio de operações internacionais para este cliente"], "justificativa": "O cliente concentra a maior parte do seu volume em pagamentos e transferências domésticas de baixo valor; a única exceção é um recebimento internacional expressivo, sinalizado pela Regra 2. Isso por si só não configura lavagem, mas justifica revisão humana antes de liberar novas remessas."}', 'tokens_entrada': 286, 'tokens_saida': 164, 'tempo_segundos': 0.05}

### Validação da saída estruturada

A resposta precisa vir com os 4 campos exigidos. Se vier malformada (JSON quebrado,
campo faltando, `nivel_risco` fora do enum), trato o erro em vez de deixar quebrar
o pipeline — em produção isso viraria um retry ou um fallback para revisão manual.

In [13]:
CAMPOS_OBRIGATORIOS = {"nivel_risco", "tipologia_suspeita", "red_flags", "justificativa"}
NIVEIS_VALIDOS = {"baixo", "medio", "alto"}

def valida_parecer(texto_bruto):
    try:
        parecer = _json.loads(texto_bruto)
    except _json.JSONDecodeError as e:
        return None, f"JSON malformado: {e}"

    faltando = CAMPOS_OBRIGATORIOS - parecer.keys()
    if faltando:
        return None, f"Campos faltando: {faltando}"

    if parecer["nivel_risco"] not in NIVEIS_VALIDOS:
        return None, f"nivel_risco inválido: {parecer['nivel_risco']!r}"

    if not isinstance(parecer["red_flags"], list):
        return None, "red_flags deveria ser uma lista"

    return parecer, None

parecer, erro = valida_parecer(resultado_v2["texto"])
if erro:
    print(f"Parecer rejeitado: {erro}")
else:
    print("Parecer válido:")
    for k, v in parecer.items():
        print(f"  {k}: {v}")

print(f"\nTokens entrada: {resultado_v2['tokens_entrada']} | tokens saída: {resultado_v2['tokens_saida']} | tempo: {resultado_v2['tempo_segundos']}s")

Parecer válido:
  nivel_risco: medio
  tipologia_suspeita: comércio internacional atípico
  red_flags: ['Recebimento único em USD muito acima do padrão do próprio cliente (5x+ a mediana)', 'Contraparte associada a operações de outros clientes no mesmo período (Zeta Importacao)', 'Ausência de histórico prévio de operações internacionais para este cliente']
  justificativa: O cliente concentra a maior parte do seu volume em pagamentos e transferências domésticas de baixo valor; a única exceção é um recebimento internacional expressivo, sinalizado pela Regra 2. Isso por si só não configura lavagem, mas justifica revisão humana antes de liberar novas remessas.

Tokens entrada: 286 | tokens saída: 164 | tempo: 0.05s


### Comparação V1 vs V2

- **V1** (prompt aberto) tende a devolver texto corrido, às vezes com opinião mais
  "genérica" e sem forçar o modelo a citar os campos que pesaram — bom para
  explorar, ruim para automatizar (não dá para validar programaticamente).
- **V2** (papel definido + formato JSON exigido + proibição explícita de
  recalcular) produz uma saída que o pipeline consegue validar e comparar em lote
  no Nível 2, e reduz a chance de a LLM "inventar" um número que já foi calculado
  em pandas. Na prática, ao rodar os dois prompts com o modelo real (fora deste
  notebook simulado), a diferença mais visível foi a taxa de parse bem-sucedido:
  V2 praticamente sempre veio em JSON válido; V1 exigia parsing manual do texto.